In [19]:
A = []

def make_feature(figure_scale, height, cast_off, plus_18, swimwear, casual_wear, otherwear, nude, dynamic_pose, release_year):
       max_height = 500
    #    max_price = 500000
       
       return [
        figure_scale, 
        height / max_height,   # Scaled 0 to ~0.85
        # price / max_price,     # Scaled 0 to ~0.45
        cast_off, plus_18, swimwear, casual_wear, otherwear, nude, dynamic_pose, 
        (release_year - 1990) / 50.0   # Scaled year
    ]

def make_data(A):
    A = []
    
    A.append(make_feature(1/7, 170, 1, 1, 0, 0, 1, 0, 0, 2012)) # Misa-nee
    A.append(make_feature(1/4, 410, 1, 0, 0, 0, 1, 0, 1, 2026)) # Xenovia
    A.append(make_feature(1/7, 150, 1, 1, 0, 0, 1, 0, 0, 2013)) # Kotone
    A.append(make_feature(1/6, 200, 1, 1, 0, 0, 1, 0, 1, 2027)) # Sino - Mimosa
    A.append(make_feature(1/8, 115, 0, 0, 0, 1, 0, 0, 0, 2013)) # Sena
    A.append(make_feature(1/5, 230, 1, 1, 0, 0, 0, 1, 1, 2023)) # Katya
    A.append(make_feature(1/6, 160, 1, 1, 0, 0, 1, 1, 0, 2025)) # Eve
    A.append(make_feature(1/10,150, 0, 0, 0, 1, 0, 0, 1, 2009)) # Tohsaka Rin
    A.append(make_feature(1/7, 190, 0, 0, 0, 0, 1, 0, 0, 2018)) # Nero
    A.append(make_feature(1/6, 160, 1, 0, 1, 0, 0, 0, 0, 2015)) # Zettai Reiko Kura Gyaru
    A.append(make_feature(1/7, 100, 1, 0, 0, 0, 1, 0, 0, 2019)) # Koneko
    A.append(make_feature(1/8, 95,  0, 0, 0, 1, 0, 0, 1, 2014)) # Koneko
    A.append(make_feature(1/7, 140, 0, 0, 1, 0, 0, 0, 0, 2018)) # Utaha
    A.append(make_feature(  0, 300, 0, 0, 1, 0, 0, 0, 0, 2027)) # Original figure
    A.append(make_feature(1/7, 220, 0, 0, 1, 0, 0, 0, 1, 2026)) # Siesta - Tante wa Mou Shindeiru
    A.append(make_feature(1/7, 200, 0, 0, 1, 0, 0, 0, 1, 2023)) # Illya
    A.append(make_feature(1/6, 265, 1, 1, 0, 0, 1, 0, 0, 2018)) # Kurara-chan
    A.append(make_feature(1/6, 250, 1, 1, 1, 0, 0, 1, 0, 2018)) # Iraha
    A.append(make_feature(1/7, 180, 0, 0, 0, 0, 0, 1, 1, 2013)) # Kuroyukihime

    return A, [1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0]

A, labels = make_data(A)
print(labels)

[1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0]


In [20]:
import random

# Initialize weights as small random floats between -0.1 and 0.1
weights = [random.uniform(-0.1, 0.1) for _ in range(len(labels))]

In [21]:
# y_hat = []
# for row in A:
#     total = 0
#     for i in range(len(row)):
#         total += row[i] * weights[i]
#     y_hat.append(total)
# y_hat

In [22]:
import math

def sigmoid(predictions:list):
    preds = []

    if not isinstance(predictions,list):
        p_clipped = max(-500, min(500, predictions))
        return(max(0,min(1,(1 / (1 + math.e ** -p_clipped)))))

    for p in range(len(predictions)):
        p_clipped = max(-500, min(500, predictions[p]))
        preds.append(max(0,min(1,(1 / (1 + math.e ** -p_clipped)))))
    return preds

# y_probabilities = sigmoid(y_hat)
# y_probabilities

In [23]:
def loss(labels, y_probabilities):
    total = 0
    eps = 1e-15 # super tiny
    for i in range(len(labels)):
        # Avoid log(0)
        p = max(eps, min(1.0 - eps, y_probabilities[i]))

        first_half = labels[i] * math.log(p)
        second_half = (1 - labels[i]) * math.log(1 - p)
        total += (first_half + second_half)
    return -total / len(labels)

In [24]:
def descend(labels, y_probabilities, A, weights, learn_rate):
    diff = []
    grad = []
    
    for i in range(len(labels)):
        diff.append(y_probabilities[i] - labels[i])

    for i in range(len(A[0])):
        total = 0
        for j in range(len(A)):
            total += A[j][i] * diff[j]
        grad.append(total)

    m = len(A)
    for i in range(len(grad)):
        grad[i] /= m
        weights[i] -= learn_rate * grad[i]

    return weights

# descend(labels, y_probabilities, A, weights, 0.001)

In [25]:
def train(epochs):
    loss_val = 9999999999
    last_loss = 99999999999
    learn_rate = .01
    print("Length of A: ",len(A))
    for epoch in range(epochs):
        y_hat = []
        
        for row in A:
            total = 0
            for i in range(len(row)):
                total += row[i] * weights[i]
            y_hat.append(total)
        y_probabilities = sigmoid(y_hat)
        loss_val = loss(labels, y_probabilities)
        # print("Loss:",loss(labels, y_probabilities))
        if loss_val < last_loss:
            last_loss = loss_val
            descend(labels, y_probabilities, A, weights, learn_rate)
        else:
            learn_rate *= .1
            print(f"Loss increased! Reducing learning rate to {learn_rate}")
            if learn_rate < .00000001:
                print(f"Learning rate too small. Stopping early at {epoch}.")
                break
            descend(labels, y_probabilities, A, weights, learn_rate)
        if epoch % 100 == 0:
            print(f"Epoch {epoch} | Loss: {loss_val}")
    print(f"Final Loss: {loss_val}")
    return loss_val

def predict(input_features):
    total = 0
    for i in range(len(input_features)):
        total += input_features[i] * weights[i]
    return sigmoid(total)

In [26]:
train(100000)

Length of A:  19
Epoch 0 | Loss: 0.6823428937901093
Epoch 100 | Loss: 0.6713199975459125
Epoch 200 | Loss: 0.6614765152077534
Epoch 300 | Loss: 0.6526498094646945
Epoch 400 | Loss: 0.6447003761592981
Epoch 500 | Loss: 0.6375089737132237
Epoch 600 | Loss: 0.630973888840109
Epoch 700 | Loss: 0.6250084374726157
Epoch 800 | Loss: 0.6195387431232838
Epoch 900 | Loss: 0.6145017993373361
Epoch 1000 | Loss: 0.6098438023720311
Epoch 1100 | Loss: 0.6055187297500418
Epoch 1200 | Loss: 0.6014871361781832
Epoch 1300 | Loss: 0.5977151379270761
Epoch 1400 | Loss: 0.5941735584362103
Epoch 1500 | Loss: 0.5908372105719936
Epoch 1600 | Loss: 0.5876842939784552
Epoch 1700 | Loss: 0.5846958889534747
Epoch 1800 | Loss: 0.5818555310648188
Epoch 1900 | Loss: 0.5791488532034196
Epoch 2000 | Loss: 0.5765632839316966
Epoch 2100 | Loss: 0.5740877928319632
Epoch 2200 | Loss: 0.5717126751207361
Epoch 2300 | Loss: 0.5694293691026184
Epoch 2400 | Loss: 0.5672303011271937
Epoch 2500 | Loss: 0.5651087536169076
Epoch 26

0.3432371301138298

In [27]:
guess = make_feature(1/6, 155, 1, 0, 0, 0, 1, 1, 1, 2014)
print(f"Guess for Rias: {predict(guess)}")

guess = make_feature(1/5, 185, 0, 0, 0, 1, 0, 0, 0, 2021)
print(f"Guess for Random: {predict(guess)}")

Guess for Rias: 0.01123167957653105
Guess for Random: 0.7336904045410988
